<a href="https://colab.research.google.com/github/Mahn17/PLN/blob/main/Tarea_4_Modelo_N_Gramas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRÁCTICA: MODELO DE LENGUAJE CON 3-GRAMAS

Equipo 8: Luis Mario Sainz Peñuñuri y Manuel Alejandro Heredia Nogales

N=3 y Stopwords on/off

# 1 Corpus

---



In [6]:
!pip install wikipedia-api

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia-api: filename=Wikipedia_API-0.9.0-py3-none-any.whl size=15422 sha256=e575976fc34daf61174202b00ce5076014d7ca4b0ccc92054e41afe99d60e5f2
  Stored in directory: /root/.cache/pip/wheels/08/22/bd/5181c75f59d48538eb0c0f3246ac541b8a3f0bce3bfd097047
Successfully built wikipedia-api


In [120]:
import wikipediaapi

USER_AGENT = "TutorialWebScraping"  # cuando haces peticiones al servidor
en_wiki = wikipediaapi.Wikipedia(USER_AGENT, language='en')
es_wiki = wikipediaapi.Wikipedia(USER_AGENT, language='es')

temas = ["cine", "inteligencia artificial", "historia", "programacion"]
all_wiki_texts = []

for tema in temas:
    page = es_wiki.page(tema)
    if page.exists():
        all_wiki_texts.append(page.text)
    else:
        print(f"El artículo '{tema}' no existe en Wikipedia en español.")

wiki_text = "\n\n".join(all_wiki_texts)

# print(wiki_text) # Comentado para evitar imprimir un texto muy largo
print(f"Se han combinado textos de {len(all_wiki_texts)} artículos de Wikipedia en 'wiki_text'.")
print(f"Longitud total de wiki_text: {len(wiki_text.split())} palabras.")

Se han combinado textos de 4 artículos de Wikipedia en 'wiki_text'.
Longitud total de wiki_text: 26442 palabras.


In [9]:
# Ruta al archivo de texto de WhatsApp
whatsapp_file_path = '/content/WhatsApp_Chat_with_Alumnos_LCC_UNISON[1].txt'

try:
    with open(whatsapp_file_path, 'r', encoding='utf-8') as f:
        whatsapp_text = f.read()
    print(f"Archivo '{whatsapp_file_path}' leído con éxito:\n{whatsapp_text}...")
except FileNotFoundError:
    print(f"Error: El archivo '{whatsapp_file_path}' no se encontró.")
except Exception as e:
    print(f"Ocurrió un error al leer el archivo: {e}")

Archivo '/content/WhatsApp_Chat_with_Alumnos_LCC_UNISON[1].txt' leído con éxito. Primeras 500 caracteres:
9/30/21, 13:54 - Messages and calls are end-to-end encrypted. Only people in this chat can read, listen to, or share them. Learn more.
7/31/20, 13:06 - ~ Martin vega created group "Alumnos LCC UNISON"
9/30/21, 13:54 - You joined using a group link.
9/30/21, 13:57 - +52 662 278 1757: <Media omitted>
9/30/21, 13:57 - +52 662 278 1757: Morros animense a participar, es un problema sencillo y habrá 4 premiados entre las categorías del más rápido, más creativo y solución más clara, asi que cualquiera ...


In [226]:
# Leer documentos propios
additional_files = [
    '/content/El-origen-del-número-cero-y-su-importancia.txt',
    '/content/RequisitosDeSoftware.txt',
    '/content/El-impacto-de-la-modernidad.txt'
]

local_texts = []
for file_path in additional_files:
    try:
        with open(file_path, 'r', encoding='latin-1') as f:
            local_texts.append(f.read())
        print(f"Archivo '{file_path}' leído con éxito.")
    except FileNotFoundError:
        print(f"Error: El archivo '{file_path}' no se encontró.")
    except Exception as e:
        print(f"Ocurrió un error al leer el archivo '{file_path}': {e}")

local_texts = "\n\n".join(local_texts)
print(f"\nSe han combinado textos de {len(additional_files)} archivos locales en 'local_texts'.")

Archivo '/content/El-origen-del-número-cero-y-su-importancia.txt' leído con éxito.
Archivo '/content/RequisitosDeSoftware.txt' leído con éxito.
Archivo '/content/El-impacto-de-la-modernidad.txt' leído con éxito.

Se han combinado textos de 3 archivos locales en 'local_texts'.


# 2 Preprocesamiento

---



In [227]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

# Descargar recursos de NLTK si no están disponibles
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

class TextPreprocessor:
    """
    Clase para el preprocesamiento de texto en español con opciones avanzadas de limpieza.

    Configuración fija utilizada para los pipelines de ejemplo:
    - Dígitos: Reemplazados por el token '<NUM>'.
    - Puntuación: Eliminada.
    - Minúsculas: Convertir todo a minúsculas.
    - Tokens de inicio/fin: Añadidos (<START>, <END>) a cada oración.
    """
    def __init__(self, remove_stopwords=False,
                 handle_digits='replace_with_token',
                 remove_punctuation=True,
                 to_lowercase=True,
                 add_start_end_tokens=True,
                 language='spanish',
                 remove_urls=False,
                 remove_emojis=False,
                 remove_whatsapp_metadata=False,
                 remove_wiki_citations=False,
                 remove_custom_artifacts=False): # Nuevo parámetro
        self.remove_stopwords = remove_stopwords
        self.handle_digits = handle_digits
        self.remove_punctuation = remove_punctuation
        self.to_lowercase = to_lowercase
        self.add_start_end_tokens = add_start_end_tokens
        self.language = language
        self.remove_urls = remove_urls
        self.remove_emojis = remove_emojis
        self.remove_whatsapp_metadata = remove_whatsapp_metadata
        self.remove_wiki_citations = remove_wiki_citations
        self.remove_custom_artifacts = remove_custom_artifacts # Guardar el nuevo parámetro

        if self.remove_stopwords:
            self.stop_words = set(stopwords.words(self.language))

        # Expresiones regulares para limpieza avanzada
        self.url_pattern = re.compile(r'https?://\S+|www\.\S+')
        self.emoji_pattern = re.compile(
            "["
            "\U0001F600-\U0001F64F"  # emoticons
            "\U0001F300-\U0001F5FF"  # symbols & pictographs
            "\U0001F680-\U0001F6FF"  # transport & map symbols
            "\U0001F1E0-\U0001F1FF"  # flags (iOS)
            "\U00002702-\U000027B0"
            "\U000024C2-\U0001F251"
            "]+", flags=re.UNICODE
        )
        self.whatsapp_date_sender_pattern = re.compile(r'^\d{1,2}/\d{1,2}/\d{2,4}, \d{1,2}:\d{2}(?: [AP]M)? - (?:[^:]+: )?')
        self.whatsapp_system_message_pattern = re.compile(r'(messages and calls are end-to-end encrypted\. only people in this chat can read, listen to, or share them\. learn more\.|<media omitted>|~?\s?\S+ created group "[^"]+"|you joined using a group link\.|.*joined using a group link\.?|joined from the community|you joined the group through a community|tap to message or add the new number|you left|left|removed|added\s\S+|this message was deleted|you deleted this message|message deleted|this message was edited|message edited)',flags=re.IGNORECASE)
        self.wiki_citation_pattern = re.compile(r'\[\d+\]|\[\s*\w+\s*\]') # [1], [2], [cita requerida]
        self.wiki_artifacts_pattern = re.compile(r'\u200b|\u200c|\u200d|\u200e|\u200f|\ufeff|\xa0|\u00ad') # Espacios de ancho cero y otros caracteres invisibles
        self.custom_artifact_pattern = re.compile(r'>>>') # Nuevo patrón para filtrar >>>


    def preprocess(self, text):
        # 1. Convertir a minúsculas
        if self.to_lowercase:
            text = text.lower()
        # 0. Limpieza inicial (antes de tokenizar en oraciones o a minúsculas)
        if self.remove_urls:
            text = self.url_pattern.sub('', text)
        if self.remove_emojis:
            text = self.emoji_pattern.sub('', text)
        if self.remove_wiki_citations:
            text = self.wiki_citation_pattern.sub('', text)
            text = self.wiki_artifacts_pattern.sub('', text)
        if self.remove_custom_artifacts: # Aplicar el nuevo filtro
            text = self.custom_artifact_pattern.sub('', text)

        if self.remove_whatsapp_metadata:
            cleaned_lines = []
            for line in text.split('\n'):
                line = self.whatsapp_date_sender_pattern.sub('', line)
                line = self.whatsapp_system_message_pattern.sub('', line)
                if line.strip():
                    cleaned_lines.append(line.strip())
            text = '\n'.join(cleaned_lines)



        # Tokenizar en oraciones para añadir tokens de inicio/fin y luego en palabras
        sentences = sent_tokenize(text, language=self.language)
        processed_sentences = []

        for sentence in sentences:
            # 2. Manejar dígitos
            if self.handle_digits == 'remove':
                sentence = re.sub(r'\d+', '', sentence)
            elif self.handle_digits == 'replace_with_token':
                sentence = re.sub(r'\d+', '<NUM>', sentence)

            # 3. Manejar puntuación
            if self.remove_punctuation:
                sentence = re.sub(r'[^\w\s<>]', ' ', sentence)
                sentence = re.sub(r'\s+', ' ', sentence).strip()

            tokens = word_tokenize(sentence, language=self.language)

            # 4. Eliminar stopwords
            if self.remove_stopwords:
                tokens = [word for word in tokens if word not in self.stop_words]

            # 5. Añadir tokens de inicio/fin
            if self.add_start_end_tokens:
                tokens = ['<START>'] + tokens + ['<END>']

            processed_sentences.append(' '.join(tokens))

        return ' '.join(processed_sentences)

# Combinar los textos de Wikipedia y WhatsApp
corpus = wiki_text + "\n\n" + whatsapp_text + "\n\n" + local_texts

print("\n" + "="*50 + "\n")
print("Texto original combinado (Wikipedia + WhatsApp):\n", corpus, "...")
print("\n" + "="*50 + "\n")

# --- Configuración de pipelines para el texto combinado ---
# Configuraciones fijas para el texto combinado:
# Dígitos: Eliminados
# Puntuación: Eliminada
# Minúsculas: Sí
# Tokens de inicio/fin: Sí
# URLs: Eliminadas
# Emojis: Eliminados
# Metadatos de WhatsApp: Eliminados (aplicable a la parte de WhatsApp)
# Citas de Wikipedia: Eliminadas (aplicable a la parte de Wikipedia)

# Pipeline 1 (Texto Combinado): Stopwords ON
preprocessor_combined_stopwords_on = TextPreprocessor(
    remove_stopwords=False,
    handle_digits='remove',
    remove_punctuation=True,
    to_lowercase=True,
    add_start_end_tokens=True,
    remove_urls=True,
    remove_emojis=True,
    remove_whatsapp_metadata=True,
    remove_wiki_citations=True,
    remove_custom_artifacts=True # Habilitar el nuevo filtro
)
corpus_stopwords_on = preprocessor_combined_stopwords_on.preprocess(corpus)
print("Pipeline 1 (Texto Combinado - Stopwords ON):\n", corpus_stopwords_on)
print("\n" + "="*50 + "\n")

# Pipeline 2 (Texto Combinado): Stopwords OFF
preprocessor_combined_stopwords_off = TextPreprocessor(
    remove_stopwords=True,
    handle_digits='remove',
    remove_punctuation=True,
    to_lowercase=True,
    add_start_end_tokens=True,
    remove_urls=True,
    remove_emojis=True,
    remove_whatsapp_metadata=True,
    remove_wiki_citations=True,
    remove_custom_artifacts=True # Habilitar el nuevo filtro
)
corpus_stopwords_off = preprocessor_combined_stopwords_off.preprocess(corpus)
print("Pipeline 2 (Texto Combinado - Stopwords OFF):\n", corpus_stopwords_off)

Se truncaron las últimas líneas 5000 del resultado de transmisión.
10/14/25, 13:16 - +52 662 341 2644: pero en tu cuenta
10/14/25, 13:16 - +52 662 341 2644: jajajaja
10/14/25, 13:16 - +52 662 341 2644: Pa q la de lcc
10/14/25, 13:16 - +52 662 296 3657: Curon
10/14/25, 13:16 - +52 662 341 2644: <Media omitted>
10/14/25, 13:16 - Gaspar LCC: ay ekis pandilla, ni pedo
10/14/25, 13:16 - +52 662 357 9360: Se organizaron mal y dejaron a muchos sin la promoción
10/14/25, 13:16 - Gaspar LCC: a eso se atienen cuando hacen fila x algo gratis
10/14/25, 13:16 - +52 662 412 0821: Y ps, el requisito para recibir los chilaquiles era subir una historia, lo cual lo hicieron casi todos sin siquiera recibir sus chilaquiles
10/14/25, 13:16 - +52 662 412 0821: Yeah
10/14/25, 13:16 - +52 631 125 1255: <Media omitted>
10/14/25, 13:16 - Gaspar LCC: a eso si esta culo ln
10/14/25, 13:17 - +52 662 412 0821: El vato dijo de que "hasta agotar existencias"
10/14/25, 13:17 - +52 662 373 9949: <Media omitted>
10/14/2

In [213]:
corpus_stopwords_off

'<START> cine abreviatura cinematógrafo cinematografía técnica arte crear proyectar metrajes conocía películas inicios <END> <START> etimológicamente palabra cinematografía neologismo creado finales siglo xix compuesto partir dos palabras griegas lado kiné significa movimiento ver cinético cinética kinesis cineteca γραφóς grafos <END> <START> ello intentaba definir concepto imagen movimiento <END> <START> habla nacimiento cine toma referencia fecha diciembre proyectaron público primeras películas realizadas hermanos auguste louis lumière memorable sesión realizada salón indio gran café parís <END> <START> entonces experimentado serie cambios varios <END> <START> lado tecnología cinematógrafo evolucionado inicios cine mudo hermanos lumière cine digital siglo xxi <END> <START> lado evolucionado lenguaje cinematográfico incluidas convenciones género surgido así distintos géneros cinematográficos <END> <START> tercer lugar evolucionado sociedad desarrollaron distintos movimientos cinematog

In [228]:
total_words = len(corpus_stopwords_off.split())
print(f"El número total de palabras en el corpus preprocesado es: {total_words}")

El número total de palabras en el corpus preprocesado es: 149717


# 3 Modelo

---



 .

In [229]:
from nltk import bigrams, trigrams
nltk.download('punkt_tab')

def crear_modelo_ngramas(texto):
    """
    Crea modelos de trigramas y bigramas desde un texto.

    texto: String con el texto de entrenamiento (ya preprocesado y tokenizado por espacios)

    Returns:
        modelo_tri: Diccionario de probabilidades de trigramas
        modelo_bi: Diccionario de probabilidades de bigramas
    """
    # Tokenizar el texto ya preprocesado
    palabras = texto.split()

    # --- Trigramas ---
    trigramas = list(trigrams(palabras))

    # Contar ocurrencias de trigramas
    conteos_tri = {}
    for w1, w2, w3 in trigramas:
        if (w1, w2) not in conteos_tri:
            conteos_tri[(w1, w2)] = {}
        if w3 not in conteos_tri[(w1, w2)]:
            conteos_tri[(w1, w2)][w3] = 0
        conteos_tri[(w1, w2)][w3] += 1

    # Convertir a probabilidades
    modelo_tri = {}
    for par, terceras in conteos_tri.items():
        total = sum(terceras.values())
        modelo_tri[par] = {palabra: cuenta/total for palabra, cuenta in terceras.items()}

    # --- Bigramas ---
    bigramas = list(bigrams(palabras))

    # Contar ocurrencias de bigramas
    conteos_bi = {}
    for w1, w2 in bigramas:
        if w1 not in conteos_bi:
            conteos_bi[w1] = {}
        if w2 not in conteos_bi[w1]:
            conteos_bi[w1][w2] = 0
        conteos_bi[w1][w2] += 1

    # Convertir a probabilidades
    modelo_bi = {}
    for w1, segundas in conteos_bi.items():
        total = sum(segundas.values())
        modelo_bi[w1] = {palabra: cuenta/total for palabra, cuenta in segundas.items()}

    return modelo_tri, modelo_bi


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [230]:
modelo_tri_off, modelo_bi_off = crear_modelo_ngramas(corpus_stopwords_off)

In [231]:
modelo_tri_on, modelo_bi_on = crear_modelo_ngramas(corpus_stopwords_on)

# 4 Generación

---



In [232]:
import random

def sample_top_p(probabilidades, p):
  """
    Selecciona una palabra usando top-p sampling (nucleus sampling).

    Args:
        probabilidades: Diccionario {palabra: probabilidad}
        p: Umbral acumulado de probabilidad (ej. 0.75, 0.9)

    Returns:
        Una palabra seleccionada aleatoriamente entre las más probables
        cuya suma de probabilidades acumuladas no excede p.
    """
  items = sorted(probabilidades.items(), key=lambda x: x[1], reverse=True)
  acumulado, seleccion = 0, []
  for palabra, probabilidad in items:
    seleccion.append((palabra, probabilidad))
    acumulado += probabilidad
    if acumulado >= p:
      break
  palabras, probabilidades = zip(*seleccion)
  return random.choices(palabras, probabilidades)[0]

def predecir_siguiente(palabra1, palabra2, modelo1, modelo2, bi):
    """
    Predice la siguiente palabra usando trigramas y, opcionalmente, a bigramas.

    Args:
        palabra1: Primera palabra del contexto
        palabra2: Segunda palabra del contexto
        modelo1: Diccionario de trigramas con probabilidades
        modelo2: Diccionario de bigramas con probabilidades
        bi: Booleano, si True activan bigramas

    Returns:
        La siguiente palabra seleccionada con top-p sampling,
        o None si no hay predicción posible.
    """
    par = (palabra1, palabra2)

    # Verificar si conocemos este par de palabras
    if par in modelo1:
        return sample_top_p(modelo1[par], 0.75)
    elif bi and palabra2 in modelo2:
        return sample_top_p(modelo2[palabra2], 0.75)
    else:
        return None

    # Obtener la palabra con mayor probabilidad
    # probabilidades = modelo[par]
    # palabra_mas_probable = max(probabilidades, key=probabilidades.get)

    # return palabra_mas_probable

def generar_texto(palabra_inicial1, palabra_inicial2, modelo1, modelo2, longitud=50, bi=False):
    """
    Genera texto palabra por palabra usando el modelo trigramas y, opcionalmente, bigramas.

    Args:
        palabra_inicial1: Primera palabra para empezar
        palabra_inicial2: Segunda palabra para empezar
        modelo1: Modelo de trigramas
        modelo2: Modelo de bigramas
        longitud: Cuántas palabras generar
        bi: Booleano, si True activan bigramas

    Returns:
        Texto generado como string
    """
    # Empezar con las dos palabras iniciales
    texto_generado = [palabra_inicial1, palabra_inicial2]

    for _ in range(longitud):
        # Usar las últimas dos palabras para predecir la siguiente
        w1 = texto_generado[-2]
        w2 = texto_generado[-1]

        siguiente = predecir_siguiente(w1, w2, modelo1, modelo2, bi)

        # Si no hay predicción, terminar
        if siguiente is None:
            break

        texto_generado.append(siguiente)

    # Unir las palabras y luego eliminar los tokens <START> y <END>
    final_text = ' '.join(texto_generado)
    final_text = final_text.replace('<START>', '').replace('<END>', '').strip()
    return final_text

In [233]:
# Palabras de inicio:
inicios = [
    ("ayer", "fue"),
    ("inteligencia", "artificial"),
    ("historia", "del"),
    ("programacion", "orientada"),
    ("cine", "y")
]

longitud_maxima = 75


Generacion con bigramas desactivados

In [234]:
for inicio1, inicio2 in inicios:
    print(f"\n--- Generando texto con '{inicio1} {inicio2}' ---")

    # Generación con modelo sin stopwords (modelo_off) - con fallback a bigramas
    texto_generado_off = generar_texto(inicio1, inicio2, modelo_tri_off, modelo_bi_off, longitud=longitud_maxima, bi=False)
    print(f"modelo off Comenzando con '{inicio1} {inicio2}':\n\t{texto_generado_off}")

    # Generación con modelo con stopwords (modelo_on) - sin fallback a bigramas (para comparar)
    texto_generado_on = generar_texto(inicio1, inicio2, modelo_tri_on, modelo_bi_on, longitud=longitud_maxima, bi=False)
    print(f"modelo on Comenzando con '{inicio1} {inicio2}':\n\t{texto_generado_on}")


--- Generando texto con 'ayer fue' ---
modelo off Comenzando con 'ayer fue':
	ayer fue
modelo on Comenzando con 'ayer fue':
	ayer fue a decirnos que en la carrera y nomas lo tienen que pedir cambio de plan   esa no soy yo yo yo no terminé de verlos todos c c subcadena vocales i i < sizeof yeomans i cout < < yeomans eduardo < > jaksjaksjajakajajajjajajakakaj y si me daban información puedes pagar con tu usuario y contraseña del portal d alumnos era con laboratorio algebra lineal i   se abrió una investigación y

--- Generando texto con 'inteligencia artificial' ---
modelo off Comenzando con 'inteligencia artificial':
	inteligencia artificial ia alienta transparentar uso ia   problema cualquier periodización hacerla coherente términos sincrónicos diacrónicos decir válida transcurso tiempo único lugar ocurre mismo tiempo creo hobby verdadero hard skill puede ser hr excelente bien temprano padrino poll saben llama objeto   buenos días   nop bueno digo forma antagonista interpretó tratan d

Generacion con bigramas activados

In [235]:
for inicio1, inicio2 in inicios:
    print(f"\n--- Generando texto con '{inicio1} {inicio2}' ---")

    # Generación con modelo sin stopwords (modelo_off) - con fallback a bigramas
    texto_generado_off = generar_texto(inicio1, inicio2, modelo_tri_off, modelo_bi_off, longitud=longitud_maxima, bi=True)
    print(f"modelo off Comenzando con '{inicio1} {inicio2}':\n\t{texto_generado_off}")

    # Generación con modelo con stopwords (modelo_on) - sin fallback a bigramas (para comparar)
    texto_generado_on = generar_texto(inicio1, inicio2, modelo_tri_on, modelo_bi_on, longitud=longitud_maxima, bi=True)
    print(f"modelo on Comenzando con '{inicio1} {inicio2}':\n\t{texto_generado_on}")


--- Generando texto con 'ayer fue' ---
modelo off Comenzando con 'ayer fue':
	ayer fue
modelo on Comenzando con 'ayer fue':
	ayer fue a la legislación relacionada con la secretaría de gobernación ubicada en versalles no   va a empezar la platica del viernes   xd de dientes   es q no te estés inventando cosas paro xddd fue peor we cuenten cuenten en pocas palabras debiste prestar atencion en la sala de chambeo del fondo del aula de chambeo del fondo del aula de chambeo del fondo del aula de chambeo del fondo del

--- Generando texto con 'inteligencia artificial' ---
modelo off Comenzando con 'inteligencia artificial':
	inteligencia artificial abarca gran variedad programas informáticos juegos estrategia ajedrez computador videojuegos   cuenta mal escroto q dio porfaaaa enunciados puso examen yeomans   si vi gracias miércoles jueves iba dar necesitados doy sticker yipee stickers one piece   tambien buscando gente ciencias computacionales ingenieria sistemas quieran hacer prácticas nope 

Se penso que no habia trigramas para "ayer" y "fue" en el modelo sin stopwords, asi que se implementaron bigramas opcionalmente para ver si generaba algo, cuando se verifico que seguia sin generar texto aun con bigrama, surgio la duda de si alguna de las 2 habia contado como stopword, y asi fue a raiz de esto tambien se agrego el sample top p para darle mas variedad al texto generado

In [221]:
palabras_a_verificar = ['fue', 'ayer']

for palabra in palabras_a_verificar:
    if palabra in stopwords.words('spanish'):
        print(f"'{palabra}' SÍ es una stopword en español.")
    else:
        print(f"'{palabra}' NO es una stopword en español.")

'fue' SÍ es una stopword en español.
'ayer' NO es una stopword en español.


# 5 Analisis y comparación

---



In [236]:
from collections import Counter
from nltk import trigrams

def estadisticas_trigramas(texto):
    palabras = texto.split()
    trigramas = list(trigrams(palabras))
    conteos = Counter(trigramas)

    # Filtrar trigramas que contengan <START> o <END>
    trigramas_filtrados = Counter()
    for trigrama, count in conteos.items():
        if '<START>' not in trigrama and '<END>' not in trigrama:
            trigramas_filtrados[trigrama] = count

    total_trigramas_unicos = len(trigramas_filtrados)
    vocabulario = set(palabras)
    tam_vocabulario = len(vocabulario)
    top10 = trigramas_filtrados.most_common(10)

    return total_trigramas_unicos, tam_vocabulario, top10

total_off, vocab_off, top10_off = estadisticas_trigramas(corpus_stopwords_off)
total_on, vocab_on, top10_on   = estadisticas_trigramas(corpus_stopwords_on)

print("Modelo OFF:")
print("Trigramas únicos (sin tokens de inicio/fin):", total_off)
print("Tamaño vocabulario:", vocab_off)
print("Top 10 trigramas (sin tokens de inicio/fin):", top10_off)

print("\nModelo ON:")
print("Trigramas únicos (sin tokens de inicio/fin):", total_on)
print("Tamaño vocabulario:", vocab_on)
print("Top 10 trigramas (sin tokens de inicio/fin):", top10_on)

Modelo OFF:
Trigramas únicos (sin tokens de inicio/fin): 115722
Tamaño vocabulario: 22619
Top 10 trigramas (sin tokens de inicio/fin): [(('horacio', 'araiza', 'gonzalez'), 55), (('felicidades', 'horacio', 'araiza'), 51), (('votes', 'option', 'votes'), 50), (('araiza', 'gonzalez', 'felicidades'), 46), (('gonzalez', 'felicidades', 'horacio'), 46), (('feliz', 'año', 'nuevo'), 42), (('alguien', 'sabe', 'si'), 39), (('fondo', 'aula', 'chambeo'), 34), (('option', 'votes', 'option'), 33), (('feliz', 'cumpleaños', 'jared'), 33)]

Modelo ON:
Trigramas únicos (sin tokens de inicio/fin): 196896
Tamaño vocabulario: 22850
Top 10 trigramas (sin tokens de inicio/fin): [(('el', 'centro', 'de'), 191), (('en', 'el', 'centro'), 140), (('centro', 'de', 'cómputo'), 133), (('centro', 'de', 'computo'), 83), (('de', 'la', 'carrera'), 76), (('para', 'los', 'que'), 70), (('en', 'el', 'cc'), 69), (('se', 'va', 'a'), 66), (('la', 'inteligencia', 'artificial'), 64), (('va', 'a', 'ser'), 63)]


# Generando texto con 'ayer fue'   

---


**modelo off Comenzando con 'ayer fue':**  
ayer fue  
**modelo on Comenzando con 'ayer fue':**  
ayer fue a revisar y no es al fondo es en alguna de las diferentes interpretaciones por ejemplo utilizando su modelo de neuronas artificiales para procesar los datos el riesgo de volverse dependientes de la oficina jajjajajajja el centro de cómputo y para evitar el grooming en lcc pero pues no siempre las materias optativas que se usan mayoritariamente técnicas de animación en la biblioteca es en el diseño ya anda el cafecito

modelo off 0/5  
modelo on 4/5

# Generando texto con 'inteligencia artificial'   

---


**modelo off Comenzando con 'inteligencia artificial':**  
	inteligencia artificial   nominado premio oscar   finales mismo año robot aún capaz caminar tener interacción alguna personas   m k jaja vato cree q adrian va levantar petición abrir materia laboratorio verdad si truncar redondear segun funcion pseint xd creo fack olvida q dije grupos secretos plebes ponganse pilas miguel intendente tarde puede ponerlo regañan pobre si pone clases sabados gracias va cómo obligatorias   option si votes option soul votes option  
**modelo on Comenzando con 'inteligencia artificial':**  
	inteligencia artificial y el premio de los derechos de autor   no se supone lo leemos entrando a clases y todo y matricula en clases en línea   o más o menos lejanos y uno de elote cierto es que si es una lástima que ya no dejan de aceptar pagos para las personas jurídicas ficticias que sin ser humanas han sido siempre muy pocos y grupos de informacion si el mundo del cine fue


modelo off 2/5  
modelo on 3/5  

# Generando texto con 'historia del'   

---


**modelo off Comenzando con 'historia del':**  
	historia del  
**modelo on Comenzando con 'historia del':**  
	historia del cine se inició con la imagen   jesús garcía   alguien sabe si si los buscan están en el centro de computo   si conocen a alguien le sale esto al iniciar sesión   si alguien quiere p en ingeniería que existen dos turnos segun yo tiene como objetivo regular y reglamentar el uso de marca que uses ola gaspaaaar no limpian la cafetera del cc pase la contraseña jajajajaja

modelo off 0/5  
modelo on 3/5

# Generando texto con 'programacion orientada'   

---


**modelo off Comenzando con 'programacion orientada':**  
	programacion orientada    
**modelo on Comenzando con 'programacion orientada':**  
	programacion orientada

0/5 ambos modelos, no generaron nada

##  Generando texto con 'cine y'   

---


**modelo off Comenzando con 'cine y':**  
	cine y  
**modelo on Comenzando con 'cine y':**  
	cine y el aprendizaje y percepción a otras más específicas también se ha rodado la película   el siguiente semestre emergencia háganle caso al cartel xfavor alguien sabe de alguien que trabajó en una caja con una de sus acciones   cuando es el pedo   esta en facebook aunq ni lo hiciste nadie se topo unos aretes perdidos en la noche mas loca de lcc no se que paso o un grupo de

modelo off 0/5  
modelo on 3/5

En general ambos textos en sus diferentes inicios se salen muy rapido del contexto, el modelo sin stopwords se le dificulta mas generar texto, inlcusive usando bigramas, sobre todo si la 2da palabra es una stopword. En general el modelo con stopwords llega a tener mas coherencia pues hila mejor las palabras generadas.  
En general el texto con stopwords es mas natural aunque el contenido del texto tienda a ser aleatorio, la forma en la que se genera es mas similar, dentro de lo que cabe, a un texto hecho por un humano.

Consideramos como la mayor limitacion del modelo la rapida perdida de contexto lo cual lleva a generar texto aleatorio  
Solo esta generando en base a los textos proporcionados (corpus), y solo refleja las frecuencias y combinaciones de este
Los trigramas menos frecuentes pueden quedar fuera del texto generado

Algunas mejoras serian:
Un corpus mas grande
Agregar 4-grama y unigramas para mas contexto y evitar que se trunque el texto cuando no hay mas trigramas ni bigramas
